# Explore the dielectric and ferroelectric response datasets

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdi-group/mace-field-tutorial/blob/main/tutorials/macefield/04_response_datasets.ipynb)

This notebook inventories the dielectric and ferroelectric datasets already shipped in this tutorial repository. It checks split sizes, label keys, atom counts, field tags, and raw response ranges, then shows the matching MACEField training options.

The notebook reports stored labels as they are. It does not rederive their electronic-structure provenance or claim that every raw polarization value is on the same branch.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

PACKAGE_URL = "https://github.com/mdi-group/mace-field.git"
TUTORIAL_URL = "https://github.com/mdi-group/mace-field-tutorial.git"


def clone_checkout(path, url, branch):
    path = Path(path)
    if not (path / ".git").is_dir():
        if path.exists() and any(path.iterdir()):
            raise RuntimeError(f"Refusing to clone over a non-empty directory: {path}")
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", branch, url, str(path)],
            check=True,
        )


if IN_COLAB:
    REPO = Path("/content/mace-field")
    TUTORIAL_REPO = Path("/content/mace-field-tutorial")
    clone_checkout(REPO, PACKAGE_URL, "clean")
    clone_checkout(TUTORIAL_REPO, TUTORIAL_URL, "main")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--editable", str(REPO)],
        check=True,
    )
else:
    CWD = Path.cwd().resolve()
    candidates = (CWD, *CWD.parents)
    TUTORIAL_REPO = next(
        (path for path in candidates
         if (path / "tutorials/macefield/README.md").is_file()),
        None,
    )
    if TUTORIAL_REPO is None:
        raise RuntimeError(
            "Run from a mace-field-tutorial checkout, or open this notebook in Colab."
        )
    package_candidates = [
        Path(os.environ["MACEFIELD_REPO"]).expanduser().resolve()
        if os.environ.get("MACEFIELD_REPO") else None,
        TUTORIAL_REPO.parent / "mace-field",
        *candidates,
    ]
    REPO = next(
        (path.resolve() for path in package_candidates
         if path is not None and (path / "mace/modules/extensions.py").is_file()),
        None,
    )
    if REPO is None:
        raise RuntimeError(
            "Clone mdi-group/mace-field beside this tutorial checkout, or set MACEFIELD_REPO."
        )
    os.environ["MACEFIELD_REPO"] = str(REPO)

os.environ["MACEFIELD_REPO"] = str(REPO)
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(TUTORIAL_REPO))
print("MACEField source:", REPO)
print("Tutorial assets:", TUTORIAL_REPO)
print("Runtime:", "Google Colab" if IN_COLAB else "local Jupyter")


## Dataset splits and label inventory

The dielectric files carry energy, forces, stress, BEC, polarizability, ionic dielectric, and piezoelectric labels. Their electric-field tag is zero. The ferroelectric files carry energy, forces, stress, electric field, and polarization labels. Separate train, validation, and test files are provided for both families.


In [ ]:
from collections import Counter
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from ase.io import iread

DATA_ROOT = TUTORIAL_REPO / "data"
SPLITS = {
    "dielectric": {
        "train": DATA_ROOT / "dielectric/dielectric_train_2048.xyz",
        "validation": DATA_ROOT / "dielectric/dielectric_valid.xyz",
        "test": DATA_ROOT / "dielectric/dielectric_test.xyz",
    },
    "ferroelectric": {
        "train": DATA_ROOT / "ferroelectric/ferroelectric_train_2040.xyz",
        "validation": DATA_ROOT / "ferroelectric/ferroelectric_valid.xyz",
        "test": DATA_ROOT / "ferroelectric/ferroelectric_test.xyz",
    },
}


def inspect_split(family, split, path):
    frame_count = 0
    atom_counts = []
    info_key_counts = Counter()
    array_key_counts = Counter()
    field_norms = []
    polarization_values = []
    first_info_shapes = {}
    first_array_shapes = {}

    for atoms in iread(str(path), index=":"):
        frame_count += 1
        atom_counts.append(len(atoms))
        info_key_counts.update(atoms.info.keys())
        array_key_counts.update(k for k in atoms.arrays if k not in {"numbers", "positions"})
        if frame_count == 1:
            first_info_shapes = {
                key: np.asarray(value).shape for key, value in atoms.info.items()
                if key.startswith("REF_")
            }
            first_array_shapes = {
                key: value.shape for key, value in atoms.arrays.items()
                if key.startswith("REF_")
            }
        field = atoms.info.get("REF_electric_field")
        if field is not None:
            field_norms.append(float(np.linalg.norm(field)))
        polarization = atoms.info.get("REF_polarisation")
        if polarization is not None:
            polarization_values.append(np.asarray(polarization, dtype=float).reshape(3))

    assert frame_count > 0, f"Empty split: {path}"
    return {
        "dataset": family,
        "split": split,
        "frames": frame_count,
        "atoms min": min(atom_counts),
        "atoms median": int(np.median(atom_counts)),
        "atoms max": max(atom_counts),
        "field norm min (V/Å)": min(field_norms) if field_norms else None,
        "field norm max (V/Å)": max(field_norms) if field_norms else None,
        "info labels": ", ".join(sorted(k for k, n in info_key_counts.items() if n == frame_count and k.startswith("REF_"))),
        "array labels": ", ".join(sorted(k for k, n in array_key_counts.items() if n == frame_count and k.startswith("REF_"))),
        "info shapes (first frame)": first_info_shapes,
        "array shapes (first frame)": first_array_shapes,
        "polarization": np.asarray(polarization_values) if polarization_values else None,
    }


summaries = [
    inspect_split(family, split, path)
    for family, paths in SPLITS.items()
    for split, path in paths.items()
]
summary_table = pd.DataFrame([
    {key: value for key, value in row.items() if key not in {"polarization", "info shapes (first frame)", "array shapes (first frame)"}}
    for row in summaries
])
display(summary_table)
for row in summaries:
    print(f"\n{row['dataset']} / {row['split']}")
    print("info tensor shapes:", row["info shapes (first frame)"])
    print("array tensor shapes:", row["array shapes (first frame)"])


## Inspect polarization branches without flattening them

The ferroelectric labels are stored under `REF_polarisation`. Periodic polarization is multivalued, so the raw component ranges can include different branches. The plot below shows the stored values directly; for training or parity comparisons, use MACEField's branch-aware `universal_field` loss and fold reference and prediction to a common branch.


In [ ]:
ferro_rows = [row for row in summaries if row["dataset"] == "ferroelectric"]
fig, ax = plt.subplots(figsize=(8, 4.5))
for row in ferro_rows:
    values = row["polarization"]
    if values is None:
        continue
    ax.scatter(values[:, 0], values[:, 2], s=12, alpha=0.45, label=row["split"])
ax.set(xlabel="stored polarization component x", ylabel="stored polarization component z",
       title="Raw ferroelectric polarization labels by split")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

for row in ferro_rows:
    values = row["polarization"]
    if values is not None:
        print(row["split"], "raw component-wise min:", values.min(axis=0),
              "max:", values.max(axis=0))


## Map stored labels to MACEField options

Use the explicit key names from these files. The standard MACEField losses use the response targets listed here; the additional ionic dielectric and piezoelectric arrays remain available in the extxyz data but are not targets in the standard field loss.

| Target | Dielectric key | Ferroelectric key | Trainer option |
| --- | --- | --- | --- |
| Energy | `REF_energy` | `REF_energy` | `--energy_key` |
| Forces | `REF_forces` | `REF_forces` | `--forces_key` |
| Stress | `REF_stress` | `REF_stress` | `--stress_key` |
| Electric field | `REF_electric_field` | `REF_electric_field` | `--electric_field_key` |
| Born effective charges | `REF_bec` | not present | `--becs_key` |
| Polarizability | `REF_polarisability` | not present | `--polarizability_key` |
| Polarization | not present | `REF_polarisation` | `--polarization_key` |

These commands show the file and label wiring. Choose loss weights and a model size for your compute budget and intended evaluation; the notebook does not start a long training run automatically.


In [ ]:
dielectric_command = [
    "mace_run_train", "--name=MACEField-dielectric", "--model=MACEField",
    "--loss=universal_field",
    "--train_file=data/dielectric/dielectric_train_2048.xyz",
    "--valid_file=data/dielectric/dielectric_valid.xyz",
    "--test_file=data/dielectric/dielectric_test.xyz",
    "--energy_key=REF_energy", "--forces_key=REF_forces", "--stress_key=REF_stress",
    "--electric_field_key=REF_electric_field", "--becs_key=REF_bec",
    "--polarizability_key=REF_polarisability",
    "--compute_forces=True", "--compute_stress=True",
    "--compute_polarization=True", "--compute_becs=True",
    "--compute_polarizability=True",
    "--device=cuda",
]
ferroelectric_command = [
    "mace_run_train", "--name=MACEField-ferroelectric", "--model=MACEField",
    "--loss=universal_field",
    "--train_file=data/ferroelectric/ferroelectric_train_2040.xyz",
    "--valid_file=data/ferroelectric/ferroelectric_valid.xyz",
    "--test_file=data/ferroelectric/ferroelectric_test.xyz",
    "--energy_key=REF_energy", "--forces_key=REF_forces", "--stress_key=REF_stress",
    "--electric_field_key=REF_electric_field", "--polarization_key=REF_polarisation",
    "--compute_forces=True", "--compute_stress=True", "--compute_polarization=True",
    "--device=cuda",
]
print("Dielectric wiring:\n", " ".join(dielectric_command))
print("\nFerroelectric wiring:\n", " ".join(ferroelectric_command))


## Next steps

1. Choose a smaller training fraction or architecture for a short Colab run.
2. Keep the supplied validation and test files fixed; do not fold the test set into training.
3. For polarization parities, branch-fold reference and prediction consistently before computing errors.
4. Check response-label units and provenance against the source calculations before interpreting errors physically.

The notebook keeps the existing split files in place so these steps are reproducible from the repository contents.
